# Brain Hemorrhage Detection — YOLOv8
**BM 480 Derin Öğrenme | 2025-2026 Bahar | Proje 3**

---
## İçindekiler
1. GPU Kontrolü & Kurulum
2. Veri Seti İndirme (Roboflow)
3. Label Doğrulama Galerisi (20 görüntü)
4. Baseline Eğitim — YOLOv8n (imgsz=640, epochs=50)
5. Ablasyon A — Görüntü Boyutu (imgsz=960)
6. Ablasyon B1 — NMS IoU Eşiği (0.50 vs 0.70)
7. Ablasyon B2 — Confidence Sweep (0.001–0.2)
8. Hata Analizi — FP / FN (En Az 6'şar Örnek)
9. Full Test — Val & Test Per-Class Metrikleri
10. Tüm Sonuçları Zip ile İndir

## 1. GPU Kontrolü & Kurulum

In [ ]:
!nvidia-smi
import torch
print(f'\nCUDA: {torch.cuda.is_available()}')
print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Yok"}')

In [ ]:
!pip install ultralytics roboflow -q
print('Kurulum tamamlandı.')

In [ ]:
import os, random, shutil, yaml
import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ultralytics import YOLO
from roboflow import Roboflow

SEED        = 42
CLASS_NAMES = ['EDH', 'IPH', 'IVH', 'SAH', 'SDH']
COLORS_BGR  = {0:(0,0,255), 1:(0,255,0), 2:(255,165,0), 3:(255,0,255), 4:(0,255,255)}
DATA_YAML   = 'dataset/data.yaml'

random.seed(SEED)
np.random.seed(SEED)
print('Kütüphaneler yüklendi. SEED =', SEED)

## 2. Veri Seti İndirme (Roboflow)

In [ ]:
rf      = Roboflow(api_key=os.environ.get('ROBOFLOW_API_KEY', 'YOUR_API_KEY'))
project = rf.workspace('yolo-v5-h7oh7').project('brain-hemorrhage-detection-8f8o0')
dataset = project.version(1).download('yolov8', location='dataset')

for split in ['train', 'valid', 'test']:
    count = len(os.listdir(f'dataset/{split}/images')) if os.path.isdir(f'dataset/{split}/images') else 0
    print(f'{split:6s}: {count} görüntü')

In [ ]:
# data.yaml'ı mutlak yol ile güncelle (Colab için gerekli)
abs_path = os.path.abspath('dataset')
data_cfg = {
    'path'  : abs_path,
    'train' : 'train/images',
    'val'   : 'valid/images',
    'test'  : 'test/images',
    'nc'    : 5,
    'names' : CLASS_NAMES
}
with open(DATA_YAML, 'w') as f:
    yaml.dump(data_cfg, f, allow_unicode=True)
print('data.yaml güncellendi →', abs_path)

## 3. Label Doğrulama Galerisi (20 Görüntü)

In [ ]:
def yolo_to_pixel(cx, cy, w, h, iw, ih):
    return (max(0, int((cx-w/2)*iw)), max(0, int((cy-h/2)*ih)),
            min(iw-1, int((cx+w/2)*iw)), min(ih-1, int((cy+h/2)*ih)))

def draw_boxes(img_path, lbl_path):
    img = cv2.imread(img_path)
    if img is None: return None, []
    ih, iw = img.shape[:2]; classes = []
    if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
        for line in open(lbl_path):
            p = line.strip().split()
            if len(p) != 5: continue
            c, cx, cy, bw, bh = int(p[0]), float(p[1]), float(p[2]), float(p[3]), float(p[4])
            x1,y1,x2,y2 = yolo_to_pixel(cx, cy, bw, bh, iw, ih)
            col = COLORS_BGR.get(c, (255,255,255))
            cv2.rectangle(img, (x1,y1), (x2,y2), col, 2)
            lbl_txt = CLASS_NAMES[c] if c < 5 else str(c)
            (tw,th),_ = cv2.getTextSize(lbl_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
            cv2.rectangle(img, (x1,y1-th-6), (x1+tw+4,y1), col, -1)
            cv2.putText(img, lbl_txt, (x1+2,y1-4), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)
            classes.append(c)
    else:
        cv2.putText(img, 'NEGATiF', (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (180,180,180), 2)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB), classes

img_dir = 'dataset/train/images'
lbl_dir = 'dataset/train/labels'
all_imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
random.seed(SEED)
selected = random.sample(all_imgs, min(20, len(all_imgs)))

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
fig.suptitle('Label Doğrulama Galerisi — Brain Hemorrhage Detection', fontsize=14, fontweight='bold')
stats = {n: 0 for n in CLASS_NAMES}; neg_count = 0

for idx, fname in enumerate(selected):
    ax = axes[idx//5][idx%5]
    img_drawn, classes = draw_boxes(
        os.path.join(img_dir, fname),
        os.path.join(lbl_dir, os.path.splitext(fname)[0]+'.txt')
    )
    if img_drawn is None: ax.axis('off'); continue
    ax.imshow(img_drawn); ax.axis('off')
    if classes:
        ax.set_title(', '.join(set(CLASS_NAMES[c] for c in classes)), fontsize=8, color='red')
        for c in classes: stats[CLASS_NAMES[c]] += 1
    else:
        ax.set_title('Negatif', fontsize=8, color='gray'); neg_count += 1

patches = [mpatches.Patch(color=np.array(COLORS_BGR[i][::-1])/255, label=CLASS_NAMES[i]) for i in range(5)]
patches.append(mpatches.Patch(color='gray', label='Negatif'))
fig.legend(handles=patches, loc='lower center', ncol=6, fontsize=9)
plt.tight_layout(rect=[0, 0.04, 1, 0.96])
os.makedirs('results/label_gallery', exist_ok=True)
plt.savefig('results/label_gallery/label_gallery.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nSınıf dağılımı:')
for k, v in stats.items(): print(f'  {k}: {v}')
print(f'  Negatif: {neg_count}')

## 4. Baseline Eğitim — YOLOv8n (imgsz=640, epochs=50)

In [ ]:
model_baseline = YOLO('yolov8n.pt')
results_baseline = model_baseline.train(
    data=DATA_YAML,
    imgsz=640,
    epochs=50,
    patience=10,
    batch=16,
    seed=SEED,
    device=0,
    project='runs/detect',
    name='baseline_640',
    exist_ok=True,
    verbose=True,
)

In [ ]:
rd      = results_baseline.results_dict
map50   = rd.get('metrics/mAP50(B)',     0)
map5095 = rd.get('metrics/mAP50-95(B)', 0)
prec    = rd.get('metrics/precision(B)', 0)
rec     = rd.get('metrics/recall(B)',    0)
f1      = 2*prec*rec / (prec+rec+1e-8)

print('='*50)
print(f'  mAP@0.5      : {map50:.4f}')
print(f'  mAP@0.5:0.95 : {map5095:.4f}')
print(f'  Precision    : {prec:.4f}')
print(f'  Recall       : {rec:.4f}')
print(f'  F1           : {f1:.4f}')
print('='*50)

os.makedirs('results/baseline', exist_ok=True)
baseline_run_dir = 'runs/detect/baseline_640'

with open('results/baseline/metrics.txt', 'w') as f:
    f.write('=== BASELINE METRİKLER (Validation Set) ===\n')
    f.write(f'Model: YOLOv8n\nimgsz: 640\nepochs: 50 (early stopping patience=10)\nseed: {SEED}\n')
    f.write(f'mAP@0.5     : {map50:.4f}\nmAP@0.5:0.95: {map5095:.4f}\n')
    f.write(f'Precision   : {prec:.4f}\nRecall      : {rec:.4f}\nF1          : {f1:.4f}\n')

for fname in ['results.png', 'BoxPR_curve.png', 'BoxF1_curve.png',
              'confusion_matrix.png', 'confusion_matrix_normalized.png',
              'val_batch0_labels.jpg', 'val_batch0_pred.jpg']:
    src = os.path.join(baseline_run_dir, fname)
    if os.path.exists(src):
        shutil.copy(src, 'results/baseline/')

best_pt = os.path.join(baseline_run_dir, 'weights', 'best.pt')
if os.path.exists(best_pt):
    shutil.copy(best_pt, 'baseline_best.pt')
    print('baseline_best.pt kopyalandı.')
print('Kaydedildi: results/baseline/')

In [ ]:
for img_path, title in [
    ('results/baseline/results.png',                     'Baseline — Eğitim Grafikleri'),
    ('results/baseline/BoxPR_curve.png',                 'Baseline — PR Eğrisi'),
    ('results/baseline/confusion_matrix_normalized.png', 'Baseline — Normalized Confusion Matrix'),
    ('results/baseline/val_batch0_pred.jpg',             'Baseline — Val Batch Tahminleri'),
]:
    if os.path.exists(img_path):
        img = plt.imread(img_path)
        plt.figure(figsize=(12, 6))
        plt.imshow(img); plt.axis('off')
        plt.title(title, fontsize=14, fontweight='bold')
        plt.tight_layout(); plt.show()
    else:
        print(f'Bulunamadı: {img_path}')

## 5. Ablasyon A — Görüntü Boyutu (imgsz=960)

In [ ]:
model_960 = YOLO('yolov8n.pt')
results_960 = model_960.train(
    data=DATA_YAML,
    imgsz=960,
    epochs=50,
    patience=10,
    batch=8,
    seed=SEED,
    device=0,
    project='runs/detect',
    name='ablation_960',
    exist_ok=True,
    verbose=True,
)

In [ ]:
rd2     = results_960.results_dict
map50_2 = rd2.get('metrics/mAP50(B)',     0)
map95_2 = rd2.get('metrics/mAP50-95(B)', 0)
prec_2  = rd2.get('metrics/precision(B)', 0)
rec_2   = rd2.get('metrics/recall(B)',    0)
f1_2    = 2*prec_2*rec_2 / (prec_2+rec_2+1e-8)

print('\n=== ABLASYON A — imgsz KARŞILAŞTIRMASI ===')
print(f'{"Metrik":<20} {"Baseline(640)":>14} {"Ablation A(960)":>16} {"Delta":>8}')
print('-'*62)
for name, b, a in [("mAP@0.5",map50,map50_2),("mAP@0.5:0.95",map5095,map95_2),
                   ("Precision",prec,prec_2),("Recall",rec,rec_2),("F1",f1,f1_2)]:
    print(f'{name:<20} {b:>14.4f} {a:>16.4f} {(a-b)/(b+1e-8)*100:>+7.1f}%')

os.makedirs('results/ablation', exist_ok=True)
with open('results/ablation/ablation_960_metrics.txt', 'w') as f:
    f.write('=== ABLASYON A METRİKLER (imgsz=960, Validation Set) ===\n')
    f.write(f'Model: YOLOv8n | imgsz: 960 | patience=10 | seed={SEED}\n')
    f.write(f'mAP@0.5     : {map50_2:.4f}\nmAP@0.5:0.95: {map95_2:.4f}\n')
    f.write(f'Precision   : {prec_2:.4f}\nRecall      : {rec_2:.4f}\nF1          : {f1_2:.4f}\n')

labels = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall', 'F1']
x = np.arange(len(labels)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x-w/2, [map50,map5095,prec,rec,f1],         w, label='Baseline (640)', alpha=0.85, color='steelblue')
b2 = ax.bar(x+w/2, [map50_2,map95_2,prec_2,rec_2,f1_2], w, label='Ablation A (960)', alpha=0.85, color='darkorange')
ax.bar_label(b1, fmt='%.3f', fontsize=8, padding=2)
ax.bar_label(b2, fmt='%.3f', fontsize=8, padding=2)
ax.set_ylim(0, 1.05); ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Değer'); ax.set_title('Ablasyon A — imgsz Karşılaştırması (640 vs 960)', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/ablation/ablation_imgsz_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Ablasyon B1 — NMS IoU Eşiği (0.50 vs 0.70)

In [ ]:
model_iou = YOLO('baseline_best.pt')
iou_rows  = []

for iou_val in [0.50, 0.70]:
    val = model_iou.val(
        data=DATA_YAML, imgsz=640, conf=0.001, iou=iou_val,
        device=0, verbose=False
    )
    p   = val.results_dict.get('metrics/precision(B)', 0)
    r   = val.results_dict.get('metrics/recall(B)',    0)
    m50 = val.results_dict.get('metrics/mAP50(B)',     0)
    m95 = val.results_dict.get('metrics/mAP50-95(B)', 0)
    f1v = 2*p*r / (p+r+1e-8)
    iou_rows.append({'IoU':iou_val, 'Precision':p, 'Recall':r, 'F1':f1v, 'mAP50':m50, 'mAP50-95':m95})
    print(f'NMS IoU={iou_val:.2f} | P={p:.4f} R={r:.4f} F1={f1v:.4f} mAP50={m50:.4f}')

df_iou = pd.DataFrame(iou_rows)
df_iou.to_csv('results/ablation/iou_sweep.csv', index=False)
print('\nAblasyon B1 tablosu:')
print(df_iou.to_string(index=False))

r1, r2 = iou_rows
metric_names = ['Precision', 'Recall', 'F1', 'mAP@0.5', 'mAP@0.5:0.95']
v1 = [r1['Precision'], r1['Recall'], r1['F1'], r1['mAP50'], r1['mAP50-95']]
v2 = [r2['Precision'], r2['Recall'], r2['F1'], r2['mAP50'], r2['mAP50-95']]
x = np.arange(len(metric_names)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x-w/2, v1, w, label='NMS IoU=0.50', alpha=0.85, color='steelblue')
b2 = ax.bar(x+w/2, v2, w, label='NMS IoU=0.70', alpha=0.85, color='coral')
ax.bar_label(b1, fmt='%.3f', fontsize=8, padding=2)
ax.bar_label(b2, fmt='%.3f', fontsize=8, padding=2)
ax.set_ylim(0, 1.05); ax.set_xticks(x); ax.set_xticklabels(metric_names)
ax.set_ylabel('Değer'); ax.set_title('Ablasyon B1 — NMS IoU Eşiği Karşılaştırması (0.50 vs 0.70)', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/ablation/ablation_iou_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Ablasyon B2 — Confidence Sweep (0.001–0.2)

In [ ]:
model_cf = YOLO('baseline_best.pt')
conf_rows = []

for conf in [0.001, 0.01, 0.05, 0.10, 0.20]:
    val = model_cf.val(
        data=DATA_YAML, conf=conf, iou=0.5,
        imgsz=640, device=0, verbose=False
    )
    p   = val.results_dict.get('metrics/precision(B)', 0)
    r   = val.results_dict.get('metrics/recall(B)',    0)
    m50 = val.results_dict.get('metrics/mAP50(B)',     0)
    m95 = val.results_dict.get('metrics/mAP50-95(B)', 0)
    f1v = 2*p*r / (p+r+1e-8)
    conf_rows.append({'conf':conf, 'Precision':p, 'Recall':r, 'F1':f1v, 'mAP50':m50, 'mAP50-95':m95})
    print(f'conf={conf:.3f} | P={p:.3f} R={r:.3f} F1={f1v:.3f} mAP50={m50:.3f}')

df_conf = pd.DataFrame(conf_rows)
best_row = df_conf.loc[df_conf['F1'].idxmax()]
print(f'\nEn iyi F1 → conf={best_row["conf"]} : F1={best_row["F1"]:.4f}')
df_conf.to_csv('results/ablation/confidence_sweep.csv', index=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(df_conf['conf'], df_conf['Precision'], 'o-', color='blue',   label='Precision')
ax1.plot(df_conf['conf'], df_conf['Recall'],    's-', color='green',  label='Recall')
ax1.plot(df_conf['conf'], df_conf['F1'],        '^-', color='red',    label='F1')
ax1.axvline(best_row['conf'], color='red', linestyle='--', alpha=0.6, label=f'Best F1 conf={best_row["conf"]}')
ax1.set_xlabel('Confidence Threshold'); ax1.set_ylabel('Değer')
ax1.set_title('P / R / F1 vs Confidence'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(df_conf['conf'], df_conf['mAP50'],    'D-', color='purple', label='mAP@0.5')
ax2.plot(df_conf['conf'], df_conf['mAP50-95'], 'v-', color='orange', label='mAP@0.5:0.95')
ax2.set_xlabel('Confidence Threshold'); ax2.set_ylabel('mAP')
ax2.set_title('mAP vs Confidence'); ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('Ablasyon B2 — Confidence Sweep', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/ablation/ablation_confidence_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Hata Analizi — FP / FN (En Az 6'şar Örnek)

| Hata Türü | Tanım |
|-----------|-------|
| **FP (False Positive)** | Model tahmin etti, eşleşen GT yok (IoU < 0.5) |
| **FN (False Negative)** | GT bbox var, model tespit edemedi (IoU < 0.5) |

- Val + Test kombinasyonu (78+36 = 114 görüntü)
- Otomatik fallback: conf=0.10 → 0.05 → 0.01 (6'şar örnek garantisi)

In [ ]:
def bbox_iou(A, B):
    xA, yA = max(A[0],B[0]), max(A[1],B[1])
    xB, yB = min(A[2],B[2]), min(A[3],B[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    if inter == 0: return 0.0
    return inter / ((A[2]-A[0])*(A[3]-A[1]) + (B[2]-B[0])*(B[3]-B[1]) - inter + 1e-8)

def find_fp_fn(lbl_path, preds, iw, ih, iou_thresh=0.5):
    gt = []
    if os.path.exists(lbl_path) and os.path.getsize(lbl_path) > 0:
        for line in open(lbl_path):
            p = line.strip().split()
            if len(p) == 5:
                c,cx,cy,bw,bh = int(p[0]),float(p[1]),float(p[2]),float(p[3]),float(p[4])
                gt.append((c, *yolo_to_pixel(cx, cy, bw, bh, iw, ih)))
    pred = []
    if preds.boxes is not None and len(preds.boxes) > 0:
        for box in preds.boxes:
            x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
            pred.append((int(box.cls[0]), float(box.conf[0]), x1, y1, x2, y2))
    matched_gt = set(); matched_pred = set()
    for pi, (pc,pconf,px1,py1,px2,py2) in enumerate(pred):
        best, bi = 0.0, -1
        for gi, (gc,gx1,gy1,gx2,gy2) in enumerate(gt):
            v = bbox_iou([px1,py1,px2,py2], [gx1,gy1,gx2,gy2])
            if v > best: best, bi = v, gi
        if best >= iou_thresh: matched_pred.add(pi); matched_gt.add(bi)
    fp = [pred[i] for i in range(len(pred)) if i not in matched_pred]
    fn = [gt[i]   for i in range(len(gt))   if i not in matched_gt]
    return fp, fn, gt, pred

def draw_fp_vis(img_bgr, fp_preds, all_gt):
    img = img_bgr.copy()
    for (c,gx1,gy1,gx2,gy2) in all_gt:
        cv2.rectangle(img, (gx1,gy1), (gx2,gy2), (0,200,0), 2)
        cv2.putText(img, f'GT:{CLASS_NAMES[c]}', (gx1, max(gy1-4,12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,200,0), 1)
    for (c,conf,px1,py1,px2,py2) in fp_preds:
        cv2.rectangle(img, (px1,py1), (px2,py2), (0,0,220), 2)
        cv2.putText(img, f'FP:{CLASS_NAMES[c]}({conf:.2f})', (px1, py2+14),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.40, (0,0,220), 1)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def draw_fn_vis(img_bgr, fn_gts, all_preds):
    img = img_bgr.copy()
    for (c,gx1,gy1,gx2,gy2) in fn_gts:
        cv2.rectangle(img, (gx1,gy1), (gx2,gy2), (0,200,0), 2)
        cv2.putText(img, f'FN:{CLASS_NAMES[c]}', (gx1, max(gy1-4,12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,200,0), 1)
    for (c,conf,px1,py1,px2,py2) in all_preds:
        cv2.rectangle(img, (px1,py1), (px2,py2), (0,0,220), 2)
        cv2.putText(img, f'P:{CLASS_NAMES[c]}({conf:.2f})', (px1, py2+14),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.40, (0,0,220), 1)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def save_grid(examples, title, out_path):
    n = len(examples)
    if n == 0: print(f'UYARI: {title} için örnek bulunamadı.'); return
    cols = min(3, n); rows = (n+cols-1)//cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*4.5))
    fig.patch.set_facecolor('black')
    fig.suptitle(title, fontsize=13, fontweight='bold', color='white')
    ax_flat = np.array(axes).flatten() if n > 1 else [axes]
    for i, (fname, vis, info) in enumerate(examples):
        ax_flat[i].imshow(vis)
        ax_flat[i].set_title(f'Örnek {i+1}\n{info}', fontsize=8, color='white')
        ax_flat[i].axis('off'); ax_flat[i].set_facecolor('black')
    for j in range(n, len(ax_flat)):
        ax_flat[j].axis('off'); ax_flat[j].set_facecolor('black')
    plt.tight_layout()
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='black')
    plt.show()
    print(f'Kaydedildi: {out_path}')

print('Hata analizi fonksiyonları hazır.')

In [ ]:
model_ea = YOLO('baseline_best.pt')
N_NEEDED = 6
os.makedirs('results/error_analysis', exist_ok=True)

IMG_DIRS = [
    ('dataset/valid/images', 'dataset/valid/labels'),
    ('dataset/test/images',  'dataset/test/labels'),
]

all_images = []
for img_dir, lbl_dir in IMG_DIRS:
    for fname in os.listdir(img_dir):
        if fname.lower().endswith(('.jpg','.jpeg','.png')):
            all_images.append((img_dir, lbl_dir, fname))

print(f'Toplam görüntü: {len(all_images)} (val+test)')

# Otomatik fallback: düşük conf → daha fazla FP
for CONF_EA in [0.10, 0.05, 0.01]:
    random.seed(SEED)
    random.shuffle(all_images)
    fp_examples, fn_examples = [], []

    for img_dir, lbl_dir, fname in all_images:
        if len(fp_examples) >= N_NEEDED and len(fn_examples) >= N_NEEDED:
            break
        img_path = os.path.join(img_dir, fname)
        lbl_path = os.path.join(lbl_dir, os.path.splitext(fname)[0]+'.txt')
        img_bgr  = cv2.imread(img_path)
        if img_bgr is None: continue
        ih, iw = img_bgr.shape[:2]
        preds = model_ea.predict(img_path, conf=CONF_EA, iou=0.5, verbose=False)[0]
        fp, fn, all_gt, all_pred = find_fp_fn(lbl_path, preds, iw, ih)
        if fp and len(fp_examples) < N_NEEDED:
            fp_examples.append((fname, draw_fp_vis(img_bgr, fp, all_gt),
                                f'FP: {len(fp)} bbox  |  GT: {len(all_gt)} bbox'))
        if fn and len(fn_examples) < N_NEEDED:
            fn_examples.append((fname, draw_fn_vis(img_bgr, fn, all_pred),
                                f'FN: {len(fn)} bbox  |  Tahmin: {len(all_pred)}'))

    print(f'conf={CONF_EA} → FP: {len(fp_examples)}, FN: {len(fn_examples)}')
    if len(fp_examples) >= N_NEEDED and len(fn_examples) >= N_NEEDED:
        print('6\'şar örnek bulundu, döngü durdu.')
        break

print(f'\nSon: FP={len(fp_examples)}, FN={len(fn_examples)}')

save_grid(
    fp_examples,
    'False Positive Örnekleri  (Yeşil=GT  |  Kırmızı=Yanlış Tahmin)',
    'results/error_analysis/false_positives.png'
)
save_grid(
    fn_examples,
    'False Negative Örnekleri  (Yeşil=Kaçırılan GT  |  Kırmızı=Mevcut Tahmin)',
    'results/error_analysis/false_negatives.png'
)

## 9. Full Test — Val & Test Per-Class Metrikleri

In [ ]:
model_ft   = YOLO('baseline_best.pt')
EVAL_CONF  = 0.001  # baseline eğitim metriklerle tutarlı
os.makedirs('results/full_test', exist_ok=True)

print('\n' + '='*55 + '\nVALİDATION SETİ\n' + '='*55)
val_m = model_ft.val(data=DATA_YAML, split='val',  imgsz=640, conf=EVAL_CONF, iou=0.5, verbose=True)

print('\n' + '='*55 + '\nTEST SETİ\n' + '='*55)
tst_m = model_ft.val(data=DATA_YAML, split='test', imgsz=640, conf=EVAL_CONF, iou=0.5, verbose=True)

def get_overall(m):
    p   = m.results_dict.get('metrics/precision(B)', 0)
    r   = m.results_dict.get('metrics/recall(B)',    0)
    m50 = m.results_dict.get('metrics/mAP50(B)',     0)
    m95 = m.results_dict.get('metrics/mAP50-95(B)', 0)
    f1  = 2*p*r/(p+r+1e-8)
    return p, r, f1, m50, m95

def get_per_class(metrics, n=5):
    def safe(arr):
        try: lst = [float(v) for v in list(arr)]; return (lst+[0.0]*n)[:n]
        except: return [0.0]*n
    return safe(metrics.box.maps), safe(metrics.box.p), safe(metrics.box.r)

vp,vr,vf,vm50,vm95 = get_overall(val_m)
tp,tr,tf,tm50,tm95 = get_overall(tst_m)
val_maps,val_ps,val_rs = get_per_class(val_m)
tst_maps,tst_ps,tst_rs = get_per_class(tst_m)

print('\n' + '='*60)
print('GENEL METRİK ÖZETİ')
print('='*60)
print(f'{"Metrik":<20} {"Val (640)":>12} {"Test (640)":>12}')
print('-'*46)
for label, v, t in [("mAP@0.5",vm50,tm50),("mAP@0.5:0.95",vm95,tm95),
                    ("Precision",vp,tp),("Recall",vr,tr),("F1",vf,tf)]:
    print(f'{label:<20} {v:>12.4f} {t:>12.4f}')

print('\n' + '='*60)
print('PER-CLASS mAP@0.5')
print('='*60)
print(f'{"Sınıf":<8} {"Val":>8} {"Test":>8}')
print('-'*26)
for i, cls in enumerate(CLASS_NAMES):
    print(f'{cls:<8} {val_maps[i]:>8.4f} {tst_maps[i]:>8.4f}')

# Per-class bar chart
x = np.arange(len(CLASS_NAMES)); w = 0.35
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Per-Class Performans — Val vs Test', fontsize=13, fontweight='bold')
for ax, vals_v, vals_t, title in [
    (axes[0], val_maps, tst_maps, 'mAP@0.5'),
    (axes[1], val_ps,  tst_ps,   'Precision'),
    (axes[2], val_rs,  tst_rs,   'Recall'),
]:
    b1 = ax.bar(x-w/2, vals_v, w, label='Val',  color='steelblue',  alpha=0.85)
    b2 = ax.bar(x+w/2, vals_t, w, label='Test', color='darkorange', alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, fontsize=10)
    ax.set_ylim(0, 1.1); ax.set_title(title); ax.legend(); ax.grid(axis='y', alpha=0.3)
    for bar in [*b1, *b2]:
        h = bar.get_height()
        if h > 0.01:
            ax.text(bar.get_x()+bar.get_width()/2, h+0.02, f'{h:.2f}',
                    ha='center', va='bottom', fontsize=7.5)
plt.tight_layout()
plt.savefig('results/full_test/per_class_val_vs_test.png', dpi=150, bbox_inches='tight')
plt.show()

# Genel özet bar chart
metrics_lbl = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall', 'F1']
val_vals    = [vm50, vm95, vp, vr, vf]
test_vals   = [tm50, tm95, tp, tr, tf]
x2 = np.arange(len(metrics_lbl)); w2 = 0.35
fig2, ax2 = plt.subplots(figsize=(11, 5))
b1 = ax2.bar(x2-w2/2, val_vals,  w2, label='Validation', color='steelblue',  alpha=0.9)
b2 = ax2.bar(x2+w2/2, test_vals, w2, label='Test',       color='darkorange', alpha=0.9)
ax2.set_xticks(x2); ax2.set_xticklabels(metrics_lbl, fontsize=11)
ax2.set_ylim(0, 1.0); ax2.set_ylabel('Değer')
ax2.set_title('Baseline Model — Genel Performans (Val vs Test)', fontsize=13, fontweight='bold')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)
for bar in [*b1, *b2]:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('results/full_test/overall_val_vs_test.png', dpi=150, bbox_inches='tight')
plt.show()

# Özet txt
with open('results/full_test/full_test_summary.txt', 'w', encoding='utf-8') as f:
    f.write('=== PROJE GENEL BAŞARI TESTİ ===\n\n')
    f.write(f'{"Metrik":<20} {"Val":>10} {"Test":>10}\n' + '-'*42 + '\n')
    for label, v, t in [("mAP@0.5",vm50,tm50),("mAP@0.5:0.95",vm95,tm95),
                        ("Precision",vp,tp),("Recall",vr,tr),("F1",vf,tf)]:
        f.write(f'{label:<20} {v:>10.4f} {t:>10.4f}\n')
    f.write('\nPer-Class mAP@0.5:\n')
    f.write(f'{"Sınıf":<8} {"Val":>8} {"Test":>8}\n' + '-'*26 + '\n')
    for i, cls in enumerate(CLASS_NAMES):
        f.write(f'{cls:<8} {val_maps[i]:>8.4f} {tst_maps[i]:>8.4f}\n')
print('\nSonuçlar kaydedildi: results/full_test/')

## 10. Tüm Sonuçları Zip ile İndir

In [ ]:
export = 'proje3_sonuclar'
os.makedirs(export, exist_ok=True)

# Model ağırlıkları
if os.path.exists('baseline_best.pt'):
    shutil.copy('baseline_best.pt', f'{export}/baseline_best.pt')

# Sonuçlar klasörü
if os.path.exists('results'):
    shutil.copytree('results', f'{export}/results', dirs_exist_ok=True)

# Python dosyaları
for fname in ['train.py', 'ablation.py', 'ablation_iou.py', 'error_analysis.py',
              'label_verify.py', 'gen_report.py', 'full_test.py', 'run_all.py',
              'app_desktop.py', 'requirements.txt']:
    if os.path.exists(fname):
        shutil.copy(fname, f'{export}/')

# Zip oluştur
shutil.make_archive('proje3_sonuclar', 'zip', '.', export)
print('Zip oluşturuldu: proje3_sonuclar.zip')

# İndir
from google.colab import files
files.download('proje3_sonuclar.zip')
print('İndirme başladı!')